# Task 3: RAG Pipeline — NOVA Product Knowledge Base

**NOVA AI Platform | Assessment Task 3**

This notebook builds and evaluates NOVA's hybrid RAG pipeline:
1. Knowledge base from product catalog, FAQs, sizing guides, ingredient guides
2. Hybrid search: BM25 (sparse) + ChromaDB (dense semantic)
3. RRF score fusion
4. Cross-encoder reranking
5. LLM answer generation with citations
6. RAGAS evaluation on 20 test Q&A pairs

**Models**: `all-MiniLM-L6-v2` (embeddings) + `ms-marco-MiniLM-L-6-v2` (reranker)
**LLM**: OpenRouter free tier

---

## Setup & Installation

In [ ]:
!pip install chromadb sentence-transformers rank-bm25 ragas openai datasets -q
print('Installation complete')

In [ ]:
import os, json, time, sys

# ── Configure API key ────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = os.getenv('GROQ_API_KEY', 'your_key_here')

MODEL = "llama-3.1-8b-instant"   # free on Groq, not decommissioned

from openai import OpenAI
llm_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

# Quick sanity check
resp = llm_client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in one word"}]
)
print(f'LLM client ready — model: {MODEL}')
print("API check:", resp.choices[0].message.content)

# If rag object already exists, patch it with the new client
try:
    rag.llm_client = llm_client
    rag.llm_model = MODEL
    print("RAG client updated to Groq")
except NameError:
    pass  # rag not built yet, it will use llm_client when created

## Download Project Files from GitHub

*If running in Colab, fetch `nova_mock_db.json` and `rag_module.py` from your GitHub repo.*

In [ ]:
# Correct raw URL for GitHub
GITHUB_RAW = "https://raw.githubusercontent.com/simran681/nova-ai-platform/main"

import urllib.request

def fetch_if_missing(filename, url):
    if not os.path.exists(filename):
        print(f'Downloading {filename}...')
        urllib.request.urlretrieve(url, filename)
        print(f'  Saved {filename}')
    else:
        print(f'  {filename} already exists')

fetch_if_missing('nova_mock_db.json', f'{GITHUB_RAW}/nova_mock_db.json')
fetch_if_missing('rag_module.py', f'{GITHUB_RAW}/rag_module.py')
print('All files ready')

## Build Knowledge Base

In [ ]:
from rag_module import NOVARAGPipeline, build_nova_knowledge_base

# Build documents from mock DB
documents = build_nova_knowledge_base('nova_mock_db.json')

print(f'\nDocument breakdown:')
from collections import Counter
types = Counter(d.metadata.get('doc_type') for d in documents)
for dtype, count in types.items():
    print(f'  {dtype}: {count}')

## Initialize RAG Pipeline & Build Index

In [ ]:
rag = NOVARAGPipeline(
    embedding_model='sentence-transformers/all-MiniLM-L6-v2',
    reranker_model='cross-encoder/ms-marco-MiniLM-L-6-v2',
    chroma_collection='nova_products',
    chroma_path='./chroma_db',
    llm_client=llm_client,
    llm_model=MODEL,
    top_k_retrieval=10,
    top_k_final=3
)

# Build hybrid index
rag.build_index(documents, reset=True)

stats = rag.get_index_stats()
print('\nIndex stats:')
for k, v in stats.items():
    print(f'  {k}: {v}')

## Test Retrieval Pipeline

In [ ]:
# Test 5 diverse queries
test_queries = [
    'Does the Glow Serum contain niacinamide?',
    'What size should I order if I am a US size 8?',
    'Which NOVA products are safe to use during pregnancy?',
    'Is the Velvet Matte Foundation vegan?',
    'What is NOVA\'s return policy?'
]

for query in test_queries:
    print(f'\n{'='*60}')
    print(f'QUERY: {query}')
    
    result = rag.query(query)
    
    print(f'ANSWER: {result.answer[:300]}...' if len(result.answer) > 300 else f'ANSWER: {result.answer}')
    print(f'SOURCES: {result.citations}')
    print(f'LATENCY: {result.latency_ms:.0f}ms')
    print(f'CHUNKS RETRIEVED: {len(result.retrieved_chunks)}')
    print(f'TOP CHUNK RERANK SCORE: {result.retrieved_chunks[0].rerank_score:.3f}')

## RAGAS Evaluation

In [ ]:
# 20 golden Q&A pairs for RAGAS evaluation
GOLDEN_QA = [
    # Ingredient queries
    {'question': 'Does the Glow Serum contain niacinamide?',
     'ground_truth': 'Check the ingredient list of the NOVA Glow Serum for niacinamide.'},
    {'question': 'What ingredients should I avoid during pregnancy?',
     'ground_truth': 'During pregnancy, avoid Retinol, high-dose Salicylic Acid, Hydroquinone, and chemical sunscreen filters. Safe alternatives include Hyaluronic Acid, Vitamin C, Niacinamide, and mineral SPF.'},
    {'question': 'Is niacinamide safe for sensitive skin?',
     'ground_truth': 'Niacinamide is generally safe for sensitive skin in standard concentrations.'},
    {'question': 'What is bakuchiol and is it a retinol alternative?',
     'ground_truth': 'Bakuchiol is a plant-based retinol alternative that is safe during pregnancy.'},
    {'question': 'Does NOVA use parabens in their skincare?',
     'ground_truth': 'Paraben information would be listed in each product\'s ingredient list.'},
    # Vegan/cruelty-free
    {'question': 'Are all NOVA products cruelty-free?',
     'ground_truth': 'Yes, all NOVA products are 100% cruelty-free and certified by Leaping Bunny.'},
    {'question': 'What ingredients make a product non-vegan?',
     'ground_truth': 'Non-vegan ingredients include Beeswax, Lanolin, Carmine, Silk/Sericin, Honey, and animal-derived Collagen.'},
    # Sizing
    {'question': 'What size should I order if I am usually a US size 8?',
     'ground_truth': 'A US size 8 typically corresponds to UK 12 / EU 40, which is size M or L in NOVA clothing. Measure your bust, waist, and hips and compare to the size guide.'},
    {'question': 'What EU shoe size is equivalent to UK size 6?',
     'ground_truth': 'UK size 6 corresponds to EU 39 in NOVA footwear.'},
    {'question': 'If I am between sizes, should I size up or down?',
     'ground_truth': 'NOVA recommends sizing up for comfort when between sizes.'},
    # Skin type
    {'question': 'What skincare ingredients are best for oily skin?',
     'ground_truth': 'For oily/acne-prone skin, look for Niacinamide, Salicylic Acid (BHA), Zinc, and lightweight Hyaluronic Acid.'},
    {'question': 'What ingredients help with dry skin?',
     'ground_truth': 'For dry skin, look for Hyaluronic Acid, Ceramides, Squalane, Glycerin, and heavier facial oils.'},
    {'question': 'What should sensitive skin avoid?',
     'ground_truth': 'Sensitive skin should avoid Fragrance, drying Alcohols, high-percentage AHA/BHA, and Retinol.'},
    # Returns/policy
    {'question': 'What is NOVA\'s return policy?',
     'ground_truth': 'NOVA offers a 30-day hassle-free return policy. Items must be unused and in original packaging. Opened skincare and makeup are final sale unless defective.'},
    {'question': 'Can I return opened skincare products?',
     'ground_truth': 'Opened skincare and makeup are generally final sale for hygiene reasons, unless the item is defective.'},
    # Shipping
    {'question': 'How long does international shipping take?',
     'ground_truth': 'International orders take 10-14 business days.'},
    {'question': 'Does NOVA offer free shipping?',
     'ground_truth': 'Orders over $50 qualify for free standard shipping.'},
    # Loyalty
    {'question': 'How does the NOVA loyalty program work?',
     'ground_truth': 'Earn 1 point per $1 spent. Tiers: Bronze (5% off), Silver (10% off + free shipping), Gold (15% off + early access), Platinum (20% off + personal stylist).'},
    # Product specific
    {'question': 'What SPF products does NOVA offer?',
     'ground_truth': 'NOVA offers SPF products in their skincare range, including sunscreens and CC creams with SPF protection.'},
    {'question': 'What payment methods does NOVA accept?',
     'ground_truth': 'NOVA accepts Visa, Mastercard, Amex, PayPal, Apple Pay, Google Pay, and Klarna (Buy Now Pay Later).'},
]

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

# Generate answers and retrieve contexts for all golden Q&A pairs
print('Running RAG on 20 golden Q&A pairs for RAGAS evaluation...')

ragas_data = {
    'question': [],
    'answer': [],
    'contexts': [],
    'ground_truth': []
}

for i, qa in enumerate(GOLDEN_QA):
    print(f'  [{i+1}/{len(GOLDEN_QA)}] {qa["question"][:60]}...')
    result = rag.query(qa['question'])
    
    ragas_data['question'].append(qa['question'])
    ragas_data['answer'].append(result.answer)
    ragas_data['contexts'].append([c.content for c in result.retrieved_chunks])
    ragas_data['ground_truth'].append(qa['ground_truth'])

ragas_dataset = Dataset.from_dict(ragas_data)
print(f'\nDataset ready: {len(ragas_dataset)} rows')

In [ ]:
# Configure RAGAS to use Groq
# Note: RAGAS internally uses the OpenAI Python SDK, which reads OPENAI_API_KEY
# and OPENAI_BASE_URL env vars. Setting them to Groq's values redirects all
# RAGAS evaluation calls to Groq — no OpenRouter is used anywhere.
import os
os.environ['OPENAI_API_KEY'] = GROQ_API_KEY
os.environ['OPENAI_BASE_URL'] = 'https://api.groq.com/openai/v1'

# Run RAGAS evaluation
print('Running RAGAS evaluation (this may take a few minutes)...')
ragas_result = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
)

print('\n=== RAGAS EVALUATION RESULTS ===')
print(ragas_result)

In [ ]:
import pandas as pd

# Convert to dataframe for display
ragas_df = ragas_result.to_pandas()
print('\nPer-question scores:')
display(ragas_df[['question', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']].round(3))

print('\nAggregate scores:')
agg = ragas_df[['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']].mean()
for metric, score in agg.items():
    status = '✅' if score >= 0.7 else '⚠️' if score >= 0.5 else '❌'
    print(f'  {status} {metric}: {score:.3f}')

In [ ]:
# Save evaluation report
agg_scores = ragas_df[['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']].mean().to_dict()

evaluation_report = {
    'task': 'Task 3 - RAG Pipeline Evaluation',
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'model': MODEL,
    'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
    'reranker_model': 'cross-encoder/ms-marco-MiniLM-L-6-v2',
    'search_strategy': 'BM25 + ChromaDB (cosine) → RRF fusion → cross-encoder rerank',
    'num_test_questions': len(GOLDEN_QA),
    'aggregate_scores': {k: round(v, 4) for k, v in agg_scores.items()},
    'per_question_scores': ragas_df[['question', 'faithfulness', 'answer_relevancy',
                                     'context_precision', 'context_recall']].round(4).to_dict('records'),
    'index_stats': rag.get_index_stats()
}

with open('evaluation_report.json', 'w') as f:
    json.dump(evaluation_report, f, indent=2)

print('Evaluation report saved to evaluation_report.json')
print(f"\nAggregate RAGAS Scores:")
for metric, score in agg_scores.items():
    print(f"  {metric}: {score:.4f}")

## Hybrid Search Ablation Study

In [ ]:
# Compare BM25 only vs Dense only vs Hybrid
query = 'What ingredients should I avoid if I have sensitive skin?'
print(f'Query: {query}\n')

# Full hybrid (default)
hybrid_result = rag.retrieve(query)
print('HYBRID (BM25 + Dense + Rerank) top chunks:')
for chunk in hybrid_result:
    print(f'  [{chunk.doc_id}] BM25={chunk.bm25_score:.2f}, Dense={chunk.dense_score:.3f}, '
          f'RRF={chunk.rrf_score:.4f}, Rerank={chunk.rerank_score:.3f}')
    print(f'    {chunk.content[:100]}...')

print(f'\nTotal retrieved: {len(hybrid_result)} chunks (top_k_final={rag.top_k_final})')